# Рекомендательная система для предсказания следующего товара
## Отборочное задание RecSys & AI

In [1]:
import json

In [2]:
def train_test_split(
    sessions: list[list[int]],
) -> tuple[list[list[int]], list[int]]:
    train_sessions = [session[:-1] for session in sessions]
    test_targets = [session[-1] for session in sessions]
    return train_sessions, test_targets
def hit_at_k(
    recommendations: list[list[int]],
    true_items: list[int],
    k: int = 10,
) -> float:
    assert len(recommendations) == len(true_items), \
        "recommendations и true_items должны совпадать по длине"

    hits = 0
    for recs, true_item in zip(recommendations, true_items):
        if true_item in recs[:k]:
            hits += 1

    return hits / len(true_items)


In [3]:
sessions = []
products_unique = []

with open("sessions.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            sessions.append(json.loads(line))

print(f"Всего сессий: {len(sessions)}")
print(f"Первая сессия: {sessions[0]}")

Всего сессий: 2565
Первая сессия: [380, 293, 262, 114, 123, 345, 335, 245, 272, 293, 233, 302, 247, 290, 260, 341, 293]


In [5]:
s = 0
counts = {}
for x in sessions:
    for r in x:
        s+=1
        if r not in products_unique:
            products_unique.append(r)
        if r in counts:
            counts[r] += 1
        else:
            counts[r] = 1
sorted_items = sorted(counts.items(), key=lambda x: x[1], reverse=True)

print("\nТоп-10 самых частых товаров:")
for i in range(10):
    item, count = sorted_items[i]
    print(f"{i+1}. Товар {item}: {count} раз")
print(f"Все товары: {s}")
print(f"Уникальные товары: {len(products_unique)}")
lengths = [len(session) for session in sessions]
print(f"Минимальная длина: {min(lengths)}")
print(f"Максимальная длина: {max(lengths)}")
print(f"Средняя длина: {sum(lengths) / len(lengths):.2f}")
lengths_sorted = sorted(lengths)
n = len(lengths_sorted)

print("Перцентили:")
print(f"10-й: {lengths_sorted[int(n * 0.1)]}")
print(f"25-й: {lengths_sorted[int(n * 0.25)]}")
print(f"50-й (медиана): {lengths_sorted[int(n * 0.5)]}")
print(f"75-й: {lengths_sorted[int(n * 0.75)]}")
print(f"90-й: {lengths_sorted[int(n * 0.9)]}")


Топ-10 самых частых товаров:
1. Товар 54: 2914 раз
2. Товар 335: 1691 раз
3. Товар 53: 1223 раз
4. Товар 114: 1067 раз
5. Товар 260: 833 раз
6. Товар 293: 736 раз
7. Товар 380: 571 раз
8. Товар 212: 510 раз
9. Товар 329: 492 раз
10. Товар 257: 450 раз
Все товары: 26843
Уникальные товары: 400
Минимальная длина: 3
Максимальная длина: 20
Средняя длина: 10.47
Перцентили:
10-й: 4
25-й: 5
50-й (медиана): 9
75-й: 16
90-й: 20


In [6]:
train_sessions, test_targets = train_test_split(sessions)

print(f"Оригинальная сессия 0: {sessions[0]}")
print(f"Обучающая часть:      {train_sessions[0]}")
print(f"Цель (что предсказать): {test_targets[0]}")

Оригинальная сессия 0: [380, 293, 262, 114, 123, 345, 335, 245, 272, 293, 233, 302, 247, 290, 260, 341, 293]
Обучающая часть:      [380, 293, 262, 114, 123, 345, 335, 245, 272, 293, 233, 302, 247, 290, 260, 341]
Цель (что предсказать): 293


In [9]:
graph = {}
count_used = 0
for el in products_unique:
    arr = []
    quantity = []
    fl = 0
    for x in sessions:
        for i in range(0, len(x) - 1):
            if x[i] == el:
                arr.append(x[i+1])
                fl = 1
    count_quantity = 0
    for u in range(0, len(arr)):
        for t in range(u, len(arr)):
            if u == t:
                count_quantity+=1
        quantity.append(count_quantity)
    for quantity_most in range(min(len(arr), 10)):
        max_number = max(quantity)
        max_index = quantity.index(max_number)
        if el not in graph:
            graph[el] = []
        graph[el].append(arr[max_index])
        quantity[max_index] = -1
    if fl == 1:
        count_used+=1 
print(f"Не были использованы {len(products_unique) - count_used} вершин")  
print(f"Пример рекомендаций для товара 54: {graph.get(54, [])[:10]}")

Не были использованы 0 вершин
Пример рекомендаций для товара 54: [293, 114, 293, 338, 275, 293, 329, 191, 327, 233]


In [15]:
popular_items = [item for item, _ in sorted_items[:10]]
baseline_recs = [popular_items for _ in train_sessions]
baseline_score = hit_at_k(baseline_recs, test_targets, k=10)
print("Точность")
print()
print(f"Бейзлайн: {baseline_score:.3f}")

all_recommendations = []

for i in range(len(train_sessions)):
    last_item = train_sessions[i][-1]  
    
    if last_item in graph and graph[last_item]:
        recs = graph[last_item][:10].copy()
    else:
        recs = popular_items.copy()
        
    recs = [r for r in recs if r != last_item]

    if len(recs) < 10:
        for p in popular_items:
            if p not in recs and p != last_item:
                recs.append(p)
            if len(recs) == 10:
                break
    
    if len(recs) < 10:
        for item in products_unique:
            if item not in recs and item != last_item:
                recs.append(item)
            if len(recs) == 10:
                break
    
    all_recommendations.append(recs)

graph_score = hit_at_k(all_recommendations, test_targets, k=10)
print(f"Моя модель: {graph_score:.3f}")

print(f"\nУлучшение относительно бейзлайна: {graph_score - baseline_score:.3f}")

last_items = [s[-1] for s in sessions]
count_last_54 = last_items.count(54)
print(f"\nСессий, где последний товар 54: {count_last_54} из {len(sessions)}")
print(f"Доля: {count_last_54/len(sessions)*100:.1f}%")



Точность

Бейзлайн: 0.384
Моя модель: 0.438

Улучшение относительно бейзлайна: 0.054

Сессий, где последний товар 54: 266 из 2565
Доля: 10.4%
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## Итог

**Бейзлайн:** 0.384  
**Моя модель:** 0.438  

**Улучшение:** +0.054 (14%)

В 10.4% сессий последним был товар 54 — бейзлайн угадывает их «бесплатно», так как 54 входит в топ-10 популярных.

Графовая модель превосходит бейзлайн благодаря учёту последнего просмотренного товара. Это подтверждает, что история пользователя содержит информацию для предсказания следующего товара.